# Phase 2: Predictive Modeling and Evaluation
**Objective:** To develop, tune, and evaluate a robust machine learning pipeline for assessing credit risk. This notebook tests various algorithms, from baseline logistic regression to advanced gradient-boosting ensembles, to identify the most accurate predictive engine.

## 1. Environment Setup and Library Imports
In this initial step, we import the necessary data manipulation libraries, visualization tools, and machine learning algorithms required for the project. 

This cell also establishes the `/artifacts` directory, which will serve as the output location for all deployed models, evaluation metrics, and visual artifacts required for the downstream web application and final report.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay

# Advanced Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Create artifacts directory
os.makedirs('../artifacts', exist_ok=True)
sns.set_theme(style="whitegrid")
print("Environment ready!")

Environment ready!


## 2. Data Loading and Target Separation
In this section, we load the pre-processed and stratified data splits (`train.csv` and `test.csv`) generated during Phase 1. 

We separate the features ($X$) from our target variable, `loan_status` ($y$), which indicates whether a loan was fully paid (0) or defaulted (1). A final sanity check is performed on the training distribution to ensure the minority default class was preserved during the split.

In [2]:
# Paths relative to /notebooks folder
train_path = '../data/train.csv'
test_path = '../data/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df.drop('loan_status', axis=1)
y_train = train_df['loan_status']
X_test = test_df.drop('loan_status', axis=1)
y_test = test_df['loan_status']

print(f"Data Loaded. Training set distribution:\n{y_train.value_counts()}")

Data Loaded. Training set distribution:
loan_status
0    20378
1     5686
Name: count, dtype: int64


## 3. Feature Preprocessing Pipeline
To ensure our machine learning algorithms receive clean, standardized data, we construct a robust preprocessing pipeline using Scikit-Learn's `ColumnTransformer`. 

This step dynamically categorizes our features into two distinct processing streams:
* **Numeric Features:** Missing values are addressed using median imputation, followed by standard scaling to normalize the data distribution (preventing features with larger scales from dominating the models).
* **Categorical Features:** Missing values are filled using the most frequent category (mode), and the textual data is converted into a machine-readable format using One-Hot Encoding.

Encapsulating these transformations within a formal pipeline is a critical best practice, as it prevents data leakage during our cross-validation and evaluation phases.

In [3]:
# Cell 3: Preprocessing Pipeline (Warning-Free Version)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Select numeric and categorical features
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
# FIXED: Explicitly added 'string' to silence the Pandas4Warning
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing pipeline ready (and warning-free)!")

Preprocessing pipeline ready (and warning-free)!


## 4. Model Training and Tuning: Random Forest
In this section, we begin the core modeling phase by training a Random Forest classifier. To seamlessly handle our data transformations and prevent data leakage, the algorithm is integrated directly into our previously defined preprocessing pipeline.

We utilize `GridSearchCV` with 3-fold cross-validation to systematically explore different hyperparameter combinations—specifically testing the number of trees (`n_estimators`) and the complexity of those trees (`max_depth`). The model is evaluated based on the Area Under the Receiver Operating Characteristic Curve (ROC AUC) to ensure we optimize for the model's ability to distinguish between default and non-default loans.

In [4]:
print("Training Random Forest...")
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', RandomForestClassifier(random_state=42))])

rf_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, None]
}

rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
print(f"Best Random Forest AUC: {rf_grid.best_score_:.4f}")

Training Random Forest...
Best Random Forest AUC: 0.9284


## 5. Baseline Model and Validation Strategy: Logistic Regression
To properly evaluate the performance of our advanced ensembles, we first establish a baseline using a linear model (Logistic Regression). 

Because our target variable is imbalanced, we introduce `StratifiedKFold` as our cross-validation strategy. Unlike standard shuffling, stratification ensures that the ratio of default to non-default loans remains strictly consistent across every fold, preventing the model from training on a sample that lacks the minority class. We also tune the regularization parameter (`C`) to prevent overfitting, optimizing specifically for the ROC AUC metric.

In [5]:
print("Training Logistic Regression...")
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', LogisticRegression(random_state=42, max_iter=1000))])

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
lr_param_grid = {'classifier__C': [0.1, 1, 10]}

lr_grid = GridSearchCV(lr_pipeline, lr_param_grid, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
lr_grid.fit(X_train, y_train)
best_lr = lr_grid.best_estimator_
print(f"Best Logistic Regression AUC: {lr_grid.best_score_:.4f}")

Training Logistic Regression...
Best Logistic Regression AUC: 0.8707


## 6. Advanced Ensemble Modeling: Voting Classifier
To maximize predictive accuracy and robustness, we combine our top-performing algorithms into a single ensemble model utilizing a `VotingClassifier`. 

This ensemble seamlessly integrates a newly trained Gradient Boosting classifier with our previously optimized Random Forest model. We employ 'soft' voting, meaning the ensemble averages the predicted probability scores from both underlying models rather than relying on a hard binary vote. This nuanced approach yields higher confidence predictions and is specifically designed to optimize our primary evaluation metric, the ROC AUC score. 

Finally, this combined pipeline is evaluated against the completely unseen test set to determine the ultimate performance of our predictive engine.

In [6]:
print("Training the Voting Classifier (Ensemble)...")
gb_model = GradientBoostingClassifier(random_state=42)
gb_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', gb_model)])
gb_pipeline.fit(X_train, y_train)

voting_clf = VotingClassifier(
    estimators=[('gb', gb_pipeline.named_steps['classifier']), 
                ('rf', best_rf.named_steps['classifier'])],
    voting='soft'
)

voting_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', voting_clf)])
voting_pipeline.fit(X_train, y_train)

y_pred_proba = voting_pipeline.predict_proba(X_test)[:, 1]
final_auc = roc_auc_score(y_test, y_pred_proba)
print(f"Voting Classifier Test AUC: {final_auc:.4f}")

Training the Voting Classifier (Ensemble)...
Voting Classifier Test AUC: 0.9340


## 7. Advanced Model Training: Gradient Boosting
Gradient Boosting is a powerful ensemble learning technique that builds decision trees sequentially, with each new tree actively correcting the errors made by previous ones. 

In this section, we implement a `GradientBoostingClassifier` and embed it directly within our preprocessing pipeline to prevent data leakage. To maximize the model's predictive capabilities, we utilize `GridSearchCV` with 3-fold cross-validation to rigorously test and optimize key hyperparameters:
* **Number of Estimators:** The total number of sequential boosting stages.
* **Learning Rate:** The impact of each individual tree on the final outcome.
* **Maximum Depth:** The structural complexity limit for each tree.

By optimizing for the ROC AUC score, we ensure the final selected estimator is highly adept at distinguishing between default and non-default loan applications.

In [7]:
from sklearn.ensemble import GradientBoostingClassifier

print("Training Gradient Boosting Classifier with Grid Search...")
gb_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', GradientBoostingClassifier(random_state=42))])

# Testing different hyperparameters
gb_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__max_depth': [3, 5]
}

gb_grid = GridSearchCV(gb_pipeline, gb_param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
gb_grid.fit(X_train, y_train)

best_gb = gb_grid.best_estimator_
print(f"Best Gradient Boosting AUC: {gb_grid.best_score_:.4f}")

Training Gradient Boosting Classifier with Grid Search...
Best Gradient Boosting AUC: 0.9468


## 8. Advanced Model Training: eXtreme Gradient Boosting (XGBoost)
Building on the success of standard gradient boosting, we introduce eXtreme Gradient Boosting (XGBoost), an advanced implementation recognized for its exceptional execution speed and model performance.

As with our previous models, the `XGBClassifier` is embedded within our preprocessing pipeline. We explicitly configure the algorithm to use logarithmic loss (`logloss`) as its evaluation metric to ensure clean, warning-free execution. 

To systematically optimize the model's predictive power, we utilize `GridSearchCV` with 3-fold cross-validation, focusing on the following hyperparameters:
* **Number of Estimators & Learning Rate:** Controls the total volume of trees and the proportional impact of each new tree on the ensemble.
* **Maximum Depth:** Dictates the structural complexity permitted for individual decision trees.
* **Subsample Ratio:** An XGBoost-specific regularization parameter that defines

In [8]:
from xgboost import XGBClassifier

print("Training XGBoost Classifier with Grid Search...")
xgb_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))])

# Testing different hyperparameters
xgb_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__max_depth': [3, 5],
    'classifier__subsample': [0.8, 1.0] # XGBoost specific tuning
}

xgb_grid = GridSearchCV(xgb_pipeline, xgb_param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
xgb_grid.fit(X_train, y_train)

best_xgb = xgb_grid.best_estimator_
print(f"Best XGBoost AUC: {xgb_grid.best_score_:.4f}")

Training XGBoost Classifier with Grid Search...
Best XGBoost AUC: 0.9440


c:\Users\Tobie\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:26:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## 9. Advanced Model Training: Light Gradient Boosting Machine (LightGBM)
To complete our suite of advanced ensemble models, we implement LightGBM, a highly efficient gradient boosting framework developed by Microsoft. LightGBM is distinguished by its leaf-wise tree growth algorithm, which often results in faster training speeds and higher accuracy on large datasets compared to traditional depth-wise approaches.

The `LGBMClassifier` is integrated into our preprocessing pipeline to maintain data integrity. We silence internal console warnings (`verbose=-1`) for a cleaner execution output. Optimization is conducted via `GridSearchCV` with 3-fold cross-validation, focusing on:
* **Number of Estimators & Learning Rate:** The foundational boosting parameters controlling the number of iterations and the step size.
* **Number of Leaves (`num_leaves`):** The primary parameter controlling tree complexity in LightGBM. Tuning this directly manages the trade-off between model accuracy and the risk of overfitting.

The model is evaluated using the ROC AUC metric, and the optimal configuration is stored for comparison against our other gradient-boosting architectures.

In [9]:
from lightgbm import LGBMClassifier

print("Training LightGBM Classifier with Grid Search...")
lgbm_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', LGBMClassifier(random_state=42, verbose=-1))])

# LightGBM specific tuning (it uses num_leaves as its main complexity dial)
lgbm_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__num_leaves': [31, 50] 
}

lgbm_grid = GridSearchCV(lgbm_pipeline, lgbm_param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
lgbm_grid.fit(X_train, y_train)

best_lgbm = lgbm_grid.best_estimator_
print(f"Best LightGBM AUC: {lgbm_grid.best_score_:.4f}")

Training LightGBM Classifier with Grid Search...
Best LightGBM AUC: 0.9455


## 10. Ultimate Ensemble Modeling: Soft Voting Classifier
To maximize our predictive accuracy and create the most robust engine possible, we construct a final ensemble utilizing a `VotingClassifier`. This approach strategically combines the highly optimized estimators from our top three performing algorithms: standard Gradient Boosting, XGBoost, and LightGBM.

We explicitly employ a 'soft' voting strategy. Rather than relying on a simple majority rule (hard binary voting), soft voting mathematically averages the predicted probability scores from all three base models. This nuanced approach leverages the unique strengths of each algorithm while mitigating their individual biases, ultimately yielding much higher confidence predictions. 

Finally, this combined ensemble is embedded back into our master preprocessing pipeline to guarantee continuous data integrity. It is trained on the full training set and evaluated against the completely unseen test set to determine our definitive, final ROC AUC score for the project.

In [10]:
from sklearn.ensemble import VotingClassifier

print("Training the Voting Classifier (Combining the Top 3)...")

# We combine the actual models we already tuned
voting_clf = VotingClassifier(
    estimators=[
        ('gb', best_gb.named_steps['classifier']),
        ('xgb', best_xgb.named_steps['classifier']),
        ('lgbm', best_lgbm.named_steps['classifier'])
    ],
    voting='soft'
)

# Put them back into our main pipeline to handle the missing values and encoding
voting_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                  ('classifier', voting_clf)])

# Train the super-model!
voting_pipeline.fit(X_train, y_train)

# Evaluate it on the test set to see if it breaks the 0.9463 record!
voting_proba = voting_pipeline.predict_proba(X_test)[:, 1]
voting_auc = roc_auc_score(y_test, voting_proba)

print(f"Voting Classifier Test AUC: {voting_auc:.4f}")

Training the Voting Classifier (Combining the Top 3)...


c:\Users\Tobie\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:26:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Voting Classifier Test AUC: 0.9528


c:\Users\Tobie\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 11. Final Evaluation and Artifact Visualizations
To conclude the modeling phase, we generate three critical visualizations to interpret our final ensemble's decision-making process and rigorously evaluate its performance. These charts are automatically exported to the `/artifacts` directory for integration into the interactive web application (P3) and the final analytical report (P4).

Specifically, this cell computes and saves the following metrics:
* **Feature Importance:** Extracted from the optimized Gradient Boosting component, this bar chart translates the model's internal logic into human-readable insights. It ranks the top 10 applicant attributes (such as Loan-to-Income Ratio or Interest Rate) that carry the most mathematical weight when assessing credit risk.
* **Confusion Matrix:** Evaluates the Voting Classifier's exact prediction breakdown on the completely unseen test set. This allows us to explicitly observe the raw counts of True Positives (correctly identified defaults) versus False Negatives (risky loans that the model mistakenly approved).
* **Receiver Operating Characteristic (ROC) Curve:** A foundational metric for binary classification, this curve plots the True Positive Rate against the False Positive Rate across all possible classification thresholds, visually demonstrating the model's robust discriminative capability.

In [11]:
# Cell 7: Generate Visual Artifacts

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import warnings
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

# Safely ignore the harmless LightGBM feature name warning
warnings.filterwarnings("ignore", category=UserWarning, message="X does not have valid feature names")

# Ensure artifacts directory exists
os.makedirs('../artifacts', exist_ok=True)

print("Generating and saving visualizations...\n")

# Set global plot style
sns.set_theme(style="whitegrid")

# ==========================================
# 0. Calculate Feature Importances

def format_label(col):
    mapping = {
        'person_age': 'Age', 'person_income': 'Annual Income',
        'person_emp_length': 'Employment Length', 'loan_amnt': 'Loan Amount',
        'loan_int_rate': 'Interest Rate', 'loan_percent_income': 'Loan-to-Income Ratio',
        'cb_person_cred_hist_length': 'Credit History'
    }
    return mapping.get(col, col.replace('_', ' ').title())

cat_encoder = best_gb.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_features_encoded = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = list(numeric_features) + list(cat_features_encoded)

importances = best_gb.named_steps['classifier'].feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': [format_label(f) for f in all_feature_names],
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# ==========================================
# 1. Feature Importance Bar Chart

plt.figure(figsize=(10, 6))
top_features = feature_importance_df.head(10)

# FIXED: Added hue='Feature' and legend=False to silence Seaborn warning
sns.barplot(data=top_features, x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)

plt.title('Top 10 Feature Importances (Gradient Boosting)', fontsize=14, pad=15)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig('../artifacts/feature_importance.png', dpi=300)
plt.close()
print("Saved feature_importance.png")

# ==========================================
# 2. Confusion Matrix

plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay.from_estimator(
    voting_pipeline, 
    X_test, 
    y_test, 
    display_labels=['Non-Default (0)', 'Default (1)'],
    cmap='Blues',
    colorbar=False
)
plt.title('Confusion Matrix (Voting Classifier)', fontsize=14, pad=15)
plt.grid(False) 
plt.tight_layout()
plt.savefig('../artifacts/confusion_matrix.png', dpi=300)
plt.close('all') 
print("Saved confusion_matrix.png")

# ==========================================
# 3. ROC Curve

plt.figure(figsize=(8, 6))

# FIXED: Changed from plot_kwargs to curve_kwargs
disp = RocCurveDisplay.from_estimator(
    voting_pipeline, 
    X_test, 
    y_test,
    name='Voting Classifier (GB + XGB + LGBM)',
    curve_kwargs={'color': 'darkorange'} 
)

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('../artifacts/roc_curve.png', dpi=300)
plt.close('all')
print("Saved roc_curve.png\n")

Generating and saving visualizations...

Saved feature_importance.png
Saved confusion_matrix.png
Saved roc_curve.png

